# Market Data Quality Audit

## tl;dr

This section is populated by the executed checks below. The audit evaluates whether the supplied
futures bar and tick files are safe to use in systematic research without cleaning.

## Context & Methods

### Key Assumptions

- Bar files should contain one OHLCV observation per ticker and minute.
- Tick files may contain multiple trades per timestamp, so timestamp uniqueness is not required.
- Timestamps are treated as timezone-naive because the exports do not include offsets; their true
  exchange/source timezone must be confirmed before cross-asset alignment.
- Contract symbols in filenames should agree with symbols inside each file.
- Checks are read-only and cover completeness, uniqueness, validity, temporal ordering, coverage,
  duplicate files, and obvious quote/price contradictions.

Source: `C:/Users/JOSHD/Downloads/Market Data` as observed on 2026-08-20.

## Data

### 1. Inventory and hash files

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

SOURCE_DIR = Path(r"C:\Users\JOSHD\Downloads\Market Data")
OUTPUT_DIR = Path(
    r"C:\Users\JOSHD\Documents\Codex\2026-08-20\referenced-chatgpt-conversation-this-is-an\outputs"
)
files = sorted(path for path in SOURCE_DIR.iterdir() if path.is_file())


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


inventory = pd.DataFrame(
    {
        "file": [path.name for path in files],
        "size_mb": [round(path.stat().st_size / 1_000_000, 2) for path in files],
        "sha256": [sha256(path) for path in files],
    }
)
inventory

,file,size_mb,sha256
0,CLU6_Export.csv,70.82,3e5332b5380ff979034219840598b7d1077b08cff9bb5e...
1,ESU6_Export.csv,10.55,c33903a274e11fe184e8c9a0a01030324770af47a490e2...
2,GCU6_Export.csv,0.08,397e2b2c93a52af4e8bdfd9895d2e27f7a426d4e3603e0...
3,HGU6_Export.csv,69.22,b9a2756737610187af69b9ed1c0e398052ecdc9ac15643...
4,NQU6_Ticks.txt,119.67,6b70c8fd1218f93fcdeeb7a6c013402dab6124185f2718...
5,ZNU6_Export.csv,77.13,c00ade45fca1c6a3d9305a1591cc7f0f24837fc8949f8e...


## Results

### 2. Profile bar and tick files

In [2]:
def profile_file(path: Path) -> dict[str, object]:
    frame = pd.read_csv(path, sep="\t", dtype=str, low_memory=False)
    first_column = frame.columns[0]
    repeated_headers = frame[first_column].eq(first_column).sum()
    frame = frame.loc[~frame[first_column].eq(first_column)].copy()
    row_count = len(frame)

    result: dict[str, object] = {
        "file": path.name,
        "kind": "ticks" if "PRICE" in frame.columns else "minute_bars",
        "rows": row_count,
        "columns": len(frame.columns),
        "repeated_headers": int(repeated_headers),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "null_cells": int(frame.isna().sum().sum()),
    }

    symbol_column = "-SYMBOL" if "-SYMBOL" in frame.columns else "-Ticker"
    date_column = "DATE" if "DATE" in frame.columns else "Date Time"
    symbols = frame[symbol_column].dropna().unique()
    parsed_dates = pd.to_datetime(frame[date_column], errors="coerce")
    result.update(
        {
            "symbols": ",".join(map(str, symbols[:5])),
            "symbol_count": len(symbols),
            "invalid_timestamps": int(parsed_dates.isna().sum()),
            "start": parsed_dates.min(),
            "end": parsed_dates.max(),
            "out_of_order": int((parsed_dates.diff().dropna() < pd.Timedelta(0)).sum()),
        }
    )

    if result["kind"] == "minute_bars":
        numeric_columns = ["Open", "High", "Low", "Last", "Volume", "Open Interest"]
        numeric = frame[numeric_columns].apply(pd.to_numeric, errors="coerce")
        result.update(
            {
                "invalid_numeric_cells": int(numeric.isna().sum().sum()),
                "duplicate_symbol_minutes": int(
                    frame.duplicated([symbol_column, date_column]).sum()
                ),
                "invalid_ohlc_rows": int(
                    (
                        (numeric["High"] < numeric[["Open", "Low", "Last"]].max(axis=1))
                        | (numeric["Low"] > numeric[["Open", "High", "Last"]].min(axis=1))
                    ).sum()
                ),
                "nonpositive_prices": int(
                    (numeric[["Open", "High", "Low", "Last"]] <= 0).any(axis=1).sum()
                ),
                "negative_volume": int((numeric["Volume"] < 0).sum()),
                "zero_volume": int((numeric["Volume"] == 0).sum()),
                "zero_open_interest_pct": round(
                    float((numeric["Open Interest"] == 0).mean() * 100), 3
                ),
                "missing_calendar_minutes": int(
                    parsed_dates.sort_values().diff().gt(pd.Timedelta(minutes=1)).sum()
                ),
            }
        )
    else:
        numeric_columns = ["PRICE", "TICKVOL", "BID", "ASK"]
        numeric = frame[numeric_columns].apply(pd.to_numeric, errors="coerce")
        result.update(
            {
                "invalid_numeric_cells": int(numeric.isna().sum().sum()),
                "nonpositive_prices": int((numeric["PRICE"] <= 0).sum()),
                "nonpositive_tick_volume": int((numeric["TICKVOL"] <= 0).sum()),
                "crossed_quotes": int((numeric["BID"] > numeric["ASK"]).sum()),
                "price_outside_quote": int(
                    (
                        (numeric["PRICE"] < numeric["BID"]) | (numeric["PRICE"] > numeric["ASK"])
                    ).sum()
                ),
                "tickvol_max": float(numeric["TICKVOL"].max()),
                "duplicate_symbol_timestamp": int(
                    frame.duplicated([symbol_column, date_column]).sum()
                ),
            }
        )
    return result


profiles = pd.DataFrame([profile_file(path) for path in files])
profiles

,file,kind,rows,columns,repeated_headers,exact_duplicate_rows,null_cells,symbols,symbol_count,invalid_timestamps,...,nonpositive_prices,negative_volume,zero_volume,zero_open_interest_pct,missing_calendar_minutes,nonpositive_tick_volume,crossed_quotes,price_outside_quote,tickvol_max,duplicate_symbol_timestamp
0,CLU6_Export.csv,minute_bars,1353404,8,0,0,0,CLU6,1,0,...,0,0.0,0.0,100.0,12532.0,NaN,NaN,NaN,NaN,NaN
1,ESU6_Export.csv,minute_bars,172745,8,0,0,0,ESU6,1,0,...,0,0.0,0.0,100.0,232.0,NaN,NaN,NaN,NaN,NaN
2,GCU6_Export.csv,minute_bars,1380,8,0,0,0,GCU6,1,0,...,0,0.0,0.0,100.0,0.0,NaN,NaN,NaN,NaN,NaN
3,HGU6_Export.csv,minute_bars,1292944,8,0,0,0,HGU6,1,0,...,0,0.0,0.0,100.0,51782.0,NaN,NaN,NaN,NaN,NaN
4,NQU6_Ticks.txt,ticks,2248160,6,364,726348,0,NQU6,1,0,...,0,NaN,NaN,NaN,NaN,170816.0,0.0,595353.0,1157042.0,1777054.0
5,ZNU6_Export.csv,minute_bars,1277027,8,0,0,0,ZNU6,1,0,...,0,0.0,0.0,100.0,62261.0,NaN,NaN,NaN,NaN,NaN


### 3. Check filename-to-symbol agreement and duplicate payloads

In [3]:
def expected_symbol(filename: str) -> str:
    return filename.split("_")[0].split(".")[0].upper()


profiles["expected_symbol"] = profiles["file"].map(expected_symbol)
profiles["filename_symbol_match"] = profiles.apply(
    lambda row: row["expected_symbol"] in str(row["symbols"]).split(","), axis=1
)
duplicate_payloads = (
    inventory.groupby("sha256")["file"].agg(list).loc[lambda values: values.map(len) > 1]
)
display(profiles[["file", "symbols", "expected_symbol", "filename_symbol_match"]])
display(duplicate_payloads.to_frame("identical_files"))

,file,symbols,expected_symbol,filename_symbol_match
0,CLU6_Export.csv,CLU6,CLU6,True
1,ESU6_Export.csv,ESU6,ESU6,True
2,GCU6_Export.csv,GCU6,GCU6,True
3,HGU6_Export.csv,HGU6,HGU6,True
4,NQU6_Ticks.txt,NQU6,NQU6,True
5,ZNU6_Export.csv,ZNU6,ZNU6,True


,identical_files
sha256,


### 4. Save compact, reusable evidence tables

In [4]:
inventory.to_csv(OUTPUT_DIR / "market_data_inventory.csv", index=False)
profiles.to_csv(OUTPUT_DIR / "market_data_quality_summary.csv", index=False)
print(f"Saved {len(inventory)} inventory rows and {len(profiles)} profile rows.")

Saved 6 inventory rows and 6 profile rows.


## Takeaways

- Review the executed profile above before using any file in a backtest.
- Files with mismatched names/symbols or identical hashes must be quarantined or renamed.
- Timestamps need a documented source timezone before assets are aligned.
- Open interest that is uniformly zero should be treated as unavailable, not as a real zero.
- Large calendar gaps can include legitimate exchange closures; session-aware completeness requires
  the vendor timezone and exchange calendar.